# 03.百度千帆文本嵌入模型

知识点讲解：QianfanEmbeddingsEndpoint 百度千帆嵌入模型

1. 为什么需要国产嵌入模型
   - 网络可达性：国内访问 OpenAI 需代理，千帆为国内直连，延迟低且稳定
   - 中文效果：Embedding-V1 基于中文语料训练，中文语义理解优于 all-MiniLM 等英文模型
   - 合规要求：数据留存在境内，满足国内数据合规与备案要求
   - 成本：千帆嵌入模型提供免费额度，中小项目基本够用
   - 如果想对接国内的文本嵌入模型提供商，可以考虑百度千帆，是目前国内生态最好，
     支持的模型最多，速度最快的 AI 应用开发平台

2. QianfanEmbeddingsEndpoint 组件
   - 位于 langchain_community.embeddings.baidu_qianfan_endpoint
   - 底层依赖 qianfan 官方 SDK：pip install qianfan
   - 同样实现 Embeddings 接口，embed_query / embed_documents 用法与 OpenAI 完全一致，
     这正是 LangChain 抽象层的价值：更换模型供应商无需改动上层业务代码
   - 由于目前百度千帆并没有单独封装到独立的包，可以直接从 langchain_community
     中导入

3. 鉴权配置（.env 文件）
   - QIANFAN_AK=你的 API Key
   - QIANFAN_SK=你的 Secret Key
   - 在百度智能云千帆控制台「应用接入」中创建应用后获取
   - 也可通过构造参数 qianfan_ak / qianfan_sk 显式传入
   - 注意：新版千帆也支持 QIANFAN_ACCESS_KEY / QIANFAN_SECRET_KEY 的 IAM 鉴权方式

4. 千帆可选嵌入模型
   - Embedding-V1  : 默认模型，384 维，中文通用场景，单条文本上限 384 token
   - bge-large-zh  : 1024 维，中文效果更优
   - bge-large-en  : 1024 维，英文场景
   - tao-8k        : 1024 维，支持 8K 长文本输入
   - 指定方式：QianfanEmbeddingsEndpoint(model="bge-large-zh")
   - 支持的模型涵盖：Embedding-V1、bge-large-zh、bge-large-en、tao-8k

5. 平台限制（重要注意事项）
   - 单次请求的文本条数上限为 16 条，超出会报错
     => LangChain 内部已做分批处理（chunk_size=16），但自行调用原生 SDK 时需注意
   - Embedding-V1 单条文本上限 384 token，超长文本必须先做文档分割
   - 存在 QPS 限流，免费额度下并发较低，批量嵌入需自行控制速率

6. 最佳实践建议
   - 中文知识库优先选择 bge-large-zh（1024 维）而非默认的 Embedding-V1
   - 务必配合 CacheBackedEmbeddings 缓存，减少调用次数以规避 QPS 限流
   - 长文本先经 RecursiveCharacterTextSplitter 切分到 384 token 以内
   - 与 OpenAI 向量不可混用：维度不同（384 vs 1536），语义空间也完全不同，
     更换模型后必须重建整个向量数据库
   - 生产环境建议用 with_retry() 包装，应对偶发限流与网络错误


In [ ]:
import dotenv
from langchain_community.embeddings.baidu_qianfan_endpoint import QianfanEmbeddingsEndpoint
dotenv.load_dotenv()
embeddings = QianfanEmbeddingsEndpoint()
query_vector = embeddings.embed_query("我叫慕小课，我喜欢打篮球游泳")
print(query_vector)
print(len(query_vector))